# Batch Prediction & Scoring

Use AzureML models to perform batch predictions on large datasets in Databricks.

## Scenarios:
1. Load AzureML model
2. Score large batches using Spark distributed processing
3. Handle predictions at scale
4. Store results back to Unity Catalog or Azure Storage

In [ ]:
import os
import json
import requests
import pandas as pd
import numpy as np
import logging
from pyspark.sql import functions as F
import mlflow

# Latest Databricks SDK (databricks-sdk)
try:
    from databricks.sdk import WorkspaceClient
    print("✓ Databricks SDK (databricks-sdk) imported successfully")
except ImportError:
    print("Note: databricks-sdk not installed. Install with: pip install databricks-sdk")
    WorkspaceClient = None

# Latest Azure ML SDK v2
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, ClientSecretCredential, EnvironmentCredential, ChainedTokenCredential
from azure.core.exceptions import HttpResponseError
print("✓ Azure ML SDK v2 (azure-ai-ml) imported successfully")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
AZUREML_ENDPOINT_URL = os.getenv("AZUREML_ENDPOINT_URL")
BATCH_SIZE = 1000

print(f"Endpoint: {AZUREML_ENDPOINT_URL}")
print(f"Batch Size: {BATCH_SIZE}")

## Step 1: Load Data for Scoring

In [ ]:
# Load data from Unity Catalog
catalog = "main"
schema = "features"
table = "customer_features"

df_score = spark.table(f"{catalog}.{schema}.{table}")

# Select only feature columns (no target)
feature_cols = [
    "customer_id", "amount", "hour", "day_of_week", "is_weekend", "avg_amount_7d"
]

df_score = df_score.select(*feature_cols)

print(f"Records to score: {df_score.count()}")
display(df_score.limit(5))

## Step 2: Authenticate to AzureML

In [ ]:
# Get authentication token
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

if all([tenant_id, client_id, client_secret]):
    credential = ClientSecretCredential(
        tenant_id=tenant_id,
        client_id=client_id,
        client_secret=client_secret
    )
else:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)

# Get token for AzureML
token = credential.get_token("https://ml.azure.com/.default").token
print("Successfully authenticated to AzureML")

## Step 3: Create Batch Scoring Function

In [ ]:
def score_batch(rows_df: pd.DataFrame) -> pd.DataFrame:
    """
    Score a batch of rows using AzureML endpoint.
    """
    feature_cols = ["amount", "hour", "day_of_week", "is_weekend", "avg_amount_7d"]
    
    # Prepare payload
    X = rows_df[feature_cols].fillna(0).values.tolist()
    
    payload = {
        "input_data": {
            "columns": feature_cols,
            "data": X
        }
    }
    
    try:
        # Call AzureML endpoint
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        response = requests.post(
            AZUREML_ENDPOINT_URL,
            headers=headers,
            json=payload,
            timeout=60
        )
        
        if response.status_code == 200:
            predictions = response.json()
            
            # Add predictions to dataframe
            if isinstance(predictions, list):
                rows_df["prediction"] = predictions
            else:
                rows_df["prediction"] = [predictions["prediction"]] * len(rows_df)
            
            rows_df["scoring_timestamp"] = pd.Timestamp.now()
        else:
            rows_df["prediction"] = np.nan
            rows_df["prediction_error"] = f"HTTP {response.status_code}"
    
    except Exception as e:
        rows_df["prediction"] = np.nan
        rows_df["prediction_error"] = str(e)
    
    return rows_df

print("Scoring function defined")

## Step 4: Batch Scoring with Pandas UDF (Distributed)

In [ ]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, DoubleType, TimestampType, StringType

# Define output schema
output_schema = StructType([
    StructField("customer_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("hour", DoubleType()),
    StructField("day_of_week", DoubleType()),
    StructField("is_weekend", DoubleType()),
    StructField("avg_amount_7d", DoubleType()),
    StructField("prediction", DoubleType()),
    StructField("scoring_timestamp", TimestampType()),
    StructField("prediction_error", StringType())
])

# Create Pandas UDF for distributed scoring
@pandas_udf(returnType=output_schema)
def score_batch_udf(iterator):
    for pdf in iterator:
        yield score_batch(pdf)

# Apply scoring function
df_scored = df_score.groupedMapInPandas(score_batch_udf, output_schema)

print("Scoring in progress...")
print(f"Scored records: {df_scored.count()}")
display(df_scored.select("customer_id", "prediction", "scoring_timestamp").limit(10))

## Step 5: Alternative - Iterator-based Scoring for Large Datasets

## Step 6: Store Predictions

In [ ]:
# Convert back to Spark DataFrame
df_predictions = spark.createDataFrame(df_results)

# Store in Unity Catalog
predictions_table = "customer_predictions"

df_predictions.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    f"{catalog}.{schema}.{predictions_table}"
)

print(f"Predictions saved to: {catalog}.{schema}.{predictions_table}")

# Display prediction statistics
display(df_predictions.select("prediction").describe())

## Step 7: Prediction Analysis

In [ ]:
# Distribution of predictions
print("Prediction Distribution:")
display(df_predictions.groupBy(F.floor(F.col("prediction")).alias("prediction_bucket")).count())

# Failed predictions analysis
print("\nFailed Predictions:")
failed = df_predictions.filter(F.col("prediction_error").isNotNull())
display(failed.groupBy("prediction_error").count())

# Timing analysis
print("\nScoring Duration:")
min_time = df_predictions.agg(F.min(F.col("scoring_timestamp"))).collect()[0][0]
max_time = df_predictions.agg(F.max(F.col("scoring_timestamp"))).collect()[0][0]
print(f"From: {min_time}")
print(f"To: {max_time}")
if min_time and max_time:
    duration = (max_time - min_time).total_seconds()
    print(f"Total duration: {duration} seconds")